# Single-View to Multi-View Demo

This tutorial demonstrates NeoVerse's capability to progressively expand a single-view video into a multi-view sequence through sequential camera trajectory transformations.

**Workflow:** Front → Left → Further Left (and separately: Front → Right → Further Right)

Each step uses the previous output as input, creating a continuous multi-view video sequence that reveals more of the scene from different angles.

## Configuration

- `use_lora`: Enable 4-step distilled LoRA for fast inference
- `num_frames`: 81 frames for smooth motion (NeoVerse default)
- `alpha_threshold`: Controls which pixels get repainted by diffusion (1.0 = repaint all masked regions)
- `static_scene`: Set to False for videos with dynamic scenes; True for static scenes
- `resize_mode`: "center_crop" preserves aspect ratio while fitting target resolution

In [ ]:
import torch
import os
import numpy as np
from torchvision.transforms import functional as F

import sys
sys.path.append("..")
from diffsynth.pipelines.wan_video_neoverse import WanVideoNeoVersePipeline
from diffsynth.utils.auxiliary import CameraTrajectory, load_video
from inference import generate_video


use_lora = True
model_path = "../models"
reconstructor_path = "../models/NeoVerse/reconstructor.ckpt"
low_vram = False
seed = 42
num_frames = 81
width = 560
height = 336
resize_mode = "center_crop"
static_scene = False
alpha_threshold = 1.0

num_inference_steps = 4 if use_lora else 50
cfg_scale = 1.0 if use_lora else 5.0

In [ ]:
lora_path = os.path.join(
    model_path,
    "NeoVerse/loras/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors"
) if use_lora else None

torch.manual_seed(seed)
np.random.seed(seed)

# Load model
print(f"Loading model from {model_path}...")
pipe = WanVideoNeoVersePipeline.from_pretrained(
    local_model_path=model_path,
    reconstructor_path=reconstructor_path,
    lora_path=lora_path,
    lora_alpha=1.0,
    device="cuda",
    torch_dtype=torch.bfloat16,
    enable_vram_management=low_vram,
)
print("Model loaded!")

# Step 1: Front View → Left View (Pan Left 25°)

Load the original video and apply a 25° left pan using the predefined trajectory.

In [ ]:
input_path = "../examples/videos/single_to_multi_case.mp4"
trajectory_file = "../examples/trajectories/front_to_left.json"
output_path = "../outputs/single_to_multi_case_left1.mp4"
prompt = "A smooth video with complete scene content. Inpaint any missing regions or margins naturally to match the surrounding scene."
images = load_video(input_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
cam_traj = CameraTrajectory.from_json(trajectory_file)
generate_video(
    pipe=pipe,
    input_video=images,
    prompt=prompt,
    negative_prompt="",
    cam_traj=cam_traj,
    output_path=output_path,
    alpha_threshold=alpha_threshold,
    static_flag=static_scene,
    seed=seed,
    cfg_scale=cfg_scale,
    num_inference_steps=num_inference_steps,
)
print(f"Done! Output saved to: {output_path}")

# Step 2: Left View → Further Left (Pan Left 25° again)

Use the output from Step 1 as input and apply another 25° left pan to expand the view further.

In [ ]:
input_path = "../outputs/single_to_multi_case_left1.mp4"
trajectory_file = "../examples/trajectories/front_to_left.json"
output_path = "../outputs/single_to_multi_case_left2.mp4"
prompt = "A smooth video with complete scene content. Inpaint any missing regions or margins naturally to match the surrounding scene."
images = load_video(input_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
cam_traj = CameraTrajectory.from_json(trajectory_file)
generate_video(
    pipe=pipe,
    input_video=images,
    prompt=prompt,
    negative_prompt="",
    cam_traj=cam_traj,
    output_path=output_path,
    alpha_threshold=alpha_threshold,
    static_flag=static_scene,
    seed=seed,
    cfg_scale=cfg_scale,
    num_inference_steps=num_inference_steps,
)
print(f"Done! Output saved to: {output_path}")

# Step 3: Front View → Right View (Pan Right 25°)

Now expand to the right side. Load the original video again and apply a 25° right pan.

In [ ]:
input_path = "../examples/videos/single_to_multi_case.mp4"
trajectory_file = "../examples/trajectories/front_to_right.json"
output_path = "../outputs/single_to_multi_case_right1.mp4"
prompt = "A smooth video with complete scene content. Inpaint any missing regions or margins naturally to match the surrounding scene."
images = load_video(input_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
cam_traj = CameraTrajectory.from_json(trajectory_file)
generate_video(
    pipe=pipe,
    input_video=images,
    prompt=prompt,
    negative_prompt="",
    cam_traj=cam_traj,
    output_path=output_path,
    alpha_threshold=alpha_threshold,
    static_flag=static_scene,
    seed=seed,
    cfg_scale=cfg_scale,
    num_inference_steps=num_inference_steps,
)
print(f"Done! Output saved to: {output_path}")

# Step 4: Right View → Further Right (Pan Right 25° again)

Use the output from Step 3 as input and apply another 25° right pan to reveal more of the scene.

In [ ]:
input_path = "../outputs/single_to_multi_case_right1.mp4"
trajectory_file = "../examples/trajectories/front_to_right.json"
output_path = "../outputs/single_to_multi_case_right2.mp4"
prompt = "A smooth video with complete scene content. Inpaint any missing regions or margins naturally to match the surrounding scene."
images = load_video(input_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
cam_traj = CameraTrajectory.from_json(trajectory_file)
generate_video(
    pipe=pipe,
    input_video=images,
    prompt=prompt,
    negative_prompt="",
    cam_traj=cam_traj,
    output_path=output_path,
    alpha_threshold=alpha_threshold,
    static_flag=static_scene,
    seed=seed,
    cfg_scale=cfg_scale,
    num_inference_steps=num_inference_steps,
)
print(f"Done! Output saved to: {output_path}")

## Results

Four expanded-view videos are generated in `../outputs/`:

- `single_to_multi_case_left1.mp4`: Front → Left
- `single_to_multi_case_left2.mp4`: Left → Further Left
- `single_to_multi_case_right1.mp4`: Front → Right
- `single_to_multi_case_right2.mp4`: Right → Further Right

Together, these outputs demonstrate how NeoVerse can progressively reveal unseen scene content from a single-view video by chaining relative camera trajectory transformations.